In [1]:
import json

match_data_list = []
with open('random_matches.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        match_data_list.append(json.loads(line.strip()))

print(f"총 {len(match_data_list)}개의 매치 불러옴.")


총 10개의 매치 불러옴.


In [2]:
def flatten_participant(match, participant):
    flat = {}
    
    # 상위 정보
    flat['matchId'] = match.get('metadata', {}).get('matchId')
    flat['gameDuration'] = match.get('info', {}).get('gameDuration')
    
    # 1단계 평면화
    for key, val in participant.items():
        if isinstance(val, dict) or isinstance(val, list):
            continue  # 중첩 구조는 별도 처리
        flat[key] = val

    # 2단계: challenges dict
    challenges = participant.get('challenges', {})
    for key, val in challenges.items():
        flat[f'challenges_{key}'] = val

    # 3단계: perks → statPerks + styles
    perks = participant.get('perks', {})
    
    statPerks = perks.get('statPerks', {})
    for key, val in statPerks.items():
        flat[f'perks_stat_{key}'] = val

    styles = perks.get('styles', [])
    for i, style in enumerate(styles):
        desc = style.get('description', f'style{i}')
        flat[f'perks_style_{i}_name'] = desc
        for j, sel in enumerate(style.get('selections', [])):
            for k, v in sel.items():
                flat[f'perks_style_{i}_sel{j}_{k}'] = v

    return flat


In [3]:
import pandas as pd
from tqdm import tqdm

rows = []

for match in tqdm(match_data_list):
    participants = match.get('info', {}).get('participants', [])
    for p in participants:
        flat_row = flatten_participant(match, p)
        rows.append(flat_row)

df_full = pd.DataFrame(rows)
print(f"✅ 총 {len(df_full)}개의 플레이어 행 생성됨.")
df_full.head()


100%|██████████| 10/10 [00:00<00:00, 1663.28it/s]

✅ 총 100개의 플레이어 행 생성됨.


,matchId,gameDuration,PlayerScore0,PlayerScore1,PlayerScore10,PlayerScore11,PlayerScore2,PlayerScore3,PlayerScore4,PlayerScore5,...,challenges_killsOnLanersEarlyJungleAsJungler,challenges_soloTurretsLategame,challenges_controlWardTimeCoverageInRiverOrEnemyHalf,challenges_highestChampionDamage,challenges_fasterSupportQuestCompletion,challenges_highestWardKills,challenges_baronBuffGoldAdvantageOverThreshold,challenges_earliestBaron,challenges_teleportTakedowns,challenges_thirdInhibitorDestroyedTime
0,KR_7611002827,1467,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,KR_7611002827,1467,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,KR_7611002827,1467,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,0.527868,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,KR_7611002827,1467,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,1.0,0.384932,1.0,NaN,NaN,NaN,NaN,NaN,NaN
4,KR_7611002827,1467,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,0.677826,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
import os

os.makedirs("data", exist_ok=True)
df_full.to_csv("data/player_full_data.csv", index=False, encoding='utf-8-sig')

print("✅ 저장 완료: data/player_full_data.csv")


✅ 저장 완료: data/player_full_data.csv
